<a href="https://colab.research.google.com/github/ergdipesh/BootCampAssignmentDP/blob/master/instructions_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
 !pip install -U unsloth
# !pip install -U "transformers>=4.48.0" "trl>=0.15.0" datasets accelerate peft bitsandbytes

In [2]:
import sys
print("Python:", sys.version)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [3]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not detected. Select a GPU runtime.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

GPU: Tesla T4
CUDA: 12.8


In [4]:
#Loading tokenizer
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024

# Unsloth loads the tokenizer and model together.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

print("Tokenizer:", MODEL_NAME)
print("Vocabulary size:", len(tokenizer))
print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Tokenizer: unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit
Vocabulary size: 151665
EOS token: <|im_end|>
PAD token: <|vision_pad|>


In [5]:
#Loading model
print("Model class:", model.__class__.__name__)
print("Loaded in 4-bit mode:", True)
print("Maximum sequence length:", MAX_SEQ_LENGTH)

parameter = next(model.parameters())
print("Model device:", parameter.device)
print("Parameter dtype:", parameter.dtype)

Model class: Qwen2ForCausalLM
Loaded in 4-bit mode: True
Maximum sequence length: 1024
Model device: cuda:0
Parameter dtype: torch.float16


In [6]:
#formatting instruction dataset
from pathlib import Path
from datasets import load_dataset

DATASET_PATH = Path("data/instruction_dataset.jsonl")

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"{DATASET_PATH} not found. Keep the data folder beside the notebook."
    )

raw_dataset = load_dataset(
    "json",
    data_files=str(DATASET_PATH),
    split="train",
)

print(raw_dataset)
print(raw_dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'response'],
    num_rows: 122
})
{'instruction': 'How many casual leaves are available each year?', 'response': 'Employees are eligible for 12 casual leaves per year.'}


In [8]:
SYSTEM_PROMPT = (
    "You are the company's internal HR Policy Assistant. "
    "Answer clearly and professionally using approved policy information. "
    "Do not invent eligibility rules, deadlines, amounts, approvals, or benefits. "
    "State when manager or HR approval is required. "
    "For employee-specific records, legal issues, discipline, medical details, "
    "payroll calculations, benefit claims, or unclear policy terms, direct the user to Human Resources."
)
def format_example(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"].strip()},
        {"role": "assistant", "content": example["response"].strip()},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"messages": messages, "text": text}

formatted_dataset = raw_dataset.map(format_example)
dataset = formatted_dataset.train_test_split(test_size=0.10, seed=42)

print(dataset)
print(dataset["train"][0]["text"])

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response', 'messages', 'text'],
        num_rows: 109
    })
    test: Dataset({
        features: ['instruction', 'response', 'messages', 'text'],
        num_rows: 13
    })
})
<|im_start|>system
You are the company's internal HR Policy Assistant. Answer clearly and professionally using approved policy information. Do not invent eligibility rules, deadlines, amounts, approvals, or benefits. State when manager or HR approval is required. For employee-specific records, legal issues, discipline, medical details, payroll calculations, benefit claims, or unclear policy terms, direct the user to Human Resources.<|im_end|>
<|im_start|>user
Can family members use my company laptop?<|im_end|>
<|im_start|>assistant
No. Company systems and devices must be protected from unauthorized access.<|im_end|>



In [9]:
def add_token_count(example):
    count = len(
        tokenizer(
            example["text"],
            add_special_tokens=False,
        )["input_ids"]
    )
    return {"token_count": count}

length_data = dataset["train"].map(add_token_count)
counts = length_data["token_count"]

print("Training examples:", len(counts))
print("Minimum tokens:", min(counts))
print("Maximum tokens:", max(counts))
print("Average tokens:", round(sum(counts) / len(counts), 2))
print("Over max length:", sum(x > MAX_SEQ_LENGTH for x in counts))

Map:   0%|          | 0/109 [00:00<?, ? examples/s]

Training examples: 109
Minimum tokens: 105
Maximum tokens: 130
Average tokens: 117.17
Over max length: 0


In [12]:
#Applying LoRA/QLoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()

Unsloth 2026.7.2 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [13]:
#baseline inference before training
def generate_answer(active_model, question, max_new_tokens=160, temperature=0.0):
    FastLanguageModel.for_inference(active_model)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to("cuda")

    generation_args = {
        "max_new_tokens": max_new_tokens,
        "repetition_penalty": 1.05,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    if temperature > 0:
        generation_args.update({
            "do_sample": True,
            "temperature": temperature,
            "top_p": 0.9,
        })
    else:
        generation_args["do_sample"] = False

    with torch.no_grad():
        output = active_model.generate(**inputs, **generation_args)

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

for question in [
    "How can I apply for sick leave?",
    "Can I work from another country without approval?",
    "What should I include with a reimbursement claim?",
]:
    print("=" * 90)
    print("QUESTION:", question)
    print("BASELINE:", generate_answer(model, question))


Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: How can I apply for sick leave?


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

BASELINE: To apply for sick leave, you should follow these steps:

1. **Review the Leave Application Form**: Obtain a copy of the leave application form from your employer. This form typically includes sections for your personal information, the reason for the leave, and any other relevant details.

2. **Submit the Application**: Once you have access to the form, you can submit it online through your employer's website or by sending a copy via email or fax.

3. **Follow Up**: After submitting the application, you will receive a confirmation email or notification. Follow up with your employer to confirm the status of your leave request.

4. **Apply for Sick Leave**: If approved, you will need to apply for sick leave at your local human resources office. You can find this information on the Human Resources section
QUESTION: Can I work from another country without approval?


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASELINE: No, you must obtain approval from your manager or human resources department before working from another country.
QUESTION: What should I include with a reimbursement claim?
BASELINE: When submitting a reimbursement claim, it is important to include all necessary documentation and information to ensure that your request is processed correctly. Here are some items you should typically include:

1. **Claim Number**: A unique identifier for your reimbursement request.
2. **Date of Claim**: The date when the claim was submitted.
3. **Amount of Reimbursement**: The total amount requested for reimbursement.
4. **Description of Expense**: A brief description of the expense covered by the reimbursement.
5. **Proof of Expense**: Any supporting documents or receipts that support the expense.
6. **Explanation of Request**: A brief explanation of why the reimbursement is needed.
7. **Manager's Signature**: If applicable, a signature from the manager who approved the reimbursement.

It is

In [14]:
#training the model
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

training_args = SFTConfig(
    output_dir="outputs/hr_policy_instruction_checkpoints",
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=2,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=training_args,
)

train_result = trainer.train()
train_result


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/109 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/13 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 109 | Num Epochs = 3 | Total steps = 21
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
10,2.395015,2.139338
20,1.177817,1.127712
21,1.185998,1.113025


Unsloth: Restored added_tokens_decoder metadata in outputs/hr_policy_instruction_checkpoints/checkpoint-10/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/hr_policy_instruction_checkpoints/checkpoint-20/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/hr_policy_instruction_checkpoints/checkpoint-21/tokenizer_config.json.


TrainOutput(global_step=21, training_loss=2.5549288533982777, metrics={'train_runtime': 46.4116, 'train_samples_per_second': 7.046, 'train_steps_per_second': 0.452, 'total_flos': 84302017182720.0, 'train_loss': 2.5549288533982777, 'epoch': 3.0})

In [16]:
print("Training metrics:")
for key, value in train_result.metrics.items():
    print(f"{key}: {value}")

print("\nValidation metrics:")
# Find the last log entry that contains evaluation metrics
eval_metrics = {}
for log_entry in trainer.state.log_history:
    if any(key.startswith("eval_") for key in log_entry):
        eval_metrics = {k: v for k, v in log_entry.items() if k.startswith("eval_")}

if eval_metrics:
    for key, value in eval_metrics.items():
        print(f"{key}: {value}")
else:
    print("No evaluation metrics found in log history.")

Training metrics:
train_runtime: 46.4116
train_samples_per_second: 7.046
train_steps_per_second: 0.452
total_flos: 84302017182720.0
train_loss: 2.5549288533982777
epoch: 3.0

Validation metrics:
eval_loss: 1.1130245923995972
eval_runtime: 1.9012
eval_samples_per_second: 6.838
eval_steps_per_second: 2.104


In [17]:
#saving adapter/model
ADAPTER_DIR = "outputs/hr_policy_instruction_lora"

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Saved adapter and tokenizer to:", ADAPTER_DIR)

Unsloth: Restored added_tokens_decoder metadata in outputs/hr_policy_instruction_lora/tokenizer_config.json.


Saved adapter and tokenizer to: outputs/hr_policy_instruction_lora


In [18]:
#runnning inference after training
FastLanguageModel.for_inference(model)

test_questions = [
    "How many casual leaves are employees eligible for each year?",
    "How do I apply for sick leave?",
    "Do I need medical documentation if I am absent for four days?",
    "Can I work from home without manager approval?",
    "What should be included in a reimbursement claim?",
    "How do I correct an incorrect attendance entry?",
    "How long is my notice period?",
    "Can you show me another employee's leave balance?",
]

for question in test_questions:
    print("=" * 100)
    print("QUESTION:")
    print(question)
    print("\nFINE-TUNED ANSWER:")
    print(generate_answer(model, question, max_new_tokens=180, temperature=0.0))
    print()

Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
How many casual leaves are employees eligible for each year?

FINE-TUNED ANSWER:


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Casual leave is up to 10 days per year.

QUESTION:
How do I apply for sick leave?

FINE-TUNED ANSWER:


Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Be specific about what was reimbursed, how much was claimed, and when it was incurred. Include the name of the expense, the amount, the department, the employee, and any applicable invoices or receipts.

QUESTION:
How do I correct an incorrect attendance entry?

FINE-TUNED ANSWER:


Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Review the record carefully. If you believe there was an error, contact Human Resources.

QUESTION:
How long is my notice period?

FINE-TUNED ANSWER:


Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Your notice period is 30 days.

QUESTION:
Can you show me another employee's leave balance?

FINE-TUNED ANSWER:
Please contact your manager or Human Resources.



In [19]:
#reload the saved adapter in a new session
from unsloth import FastLanguageModel

reloaded_model, reloaded_tokenizer = FastLanguageModel.from_pretrained(
    model_name="outputs/hr_policy_instruction_lora",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

FastLanguageModel.for_inference(reloaded_model)
print("Saved adapter reloaded.")

==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Saved adapter reloaded.


In [20]:
def answer_with_reloaded_model(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    prompt = reloaded_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = reloaded_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to("cuda")

    with torch.no_grad():
        output = reloaded_model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=reloaded_tokenizer.eos_token_id,
            eos_token_id=reloaded_tokenizer.eos_token_id,
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return reloaded_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

print(answer_with_reloaded_model(
    "What should I do if I cannot access the HR portal while I am sick?"
))

Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Contact your manager or Human Resources.
